In [ ]:
## Jędrzej Sokołowski

In [4]:
import pandas as pd
import os

In [ ]:
# You will need the downloaded gtfs folder in the working directory

In [6]:
def load_gtfs_file(filename, use_cols=None, drop_cols=None):
    filepath = os.path.join("gtfs", filename)
    df = pd.read_csv(filepath, usecols=use_cols)
    if drop_cols:
        df = df.drop(columns=drop_cols, errors='ignore')
    return df

def clean_and_process_gtfs_data():
    df_cal = load_gtfs_file("calendar.txt", use_cols=["service_id", "start_date", "end_date"])
    df_trips = load_gtfs_file("trips.txt", drop_cols=["trip_headsign", "direction_id", "shape_id"])
    df_routes = load_gtfs_file("routes.txt", drop_cols=["agency_id", "route_long_name", "route_desc"])
    df_stop_times = load_gtfs_file("stop_times.txt", drop_cols=["stop_headsign", "pickup_type", "drop_off_type", "timepoint"])
    df_stops = load_gtfs_file("stops.txt", drop_cols=["stop_code"])

    df = (
        df_cal
        .merge(df_trips, on="service_id")
        .merge(df_routes, on="route_id")
        .merge(df_stop_times, on="trip_id")
        .merge(df_stops, on="stop_id")
    )

    # Remove times >= '24:00:00' to avoid ValueError
    df = df[df["arrival_time"] < '24:00:00']
    df = df[df["departure_time"] < '24:00:00']

    df["start_date"] = pd.to_datetime(df["start_date"], format="%Y%m%d")
    df["end_date"] = pd.to_datetime(df["end_date"], format="%Y%m%d")
    df["arrival_time"] = pd.to_datetime(df["arrival_time"], format="%H:%M:%S").dt.time
    df["departure_time"] = pd.to_datetime(df["departure_time"], format="%H:%M:%S").dt.time

    # Combine date and time to create full datetime objects
    df['arrival_datetime'] = pd.to_datetime(df['start_date'].dt.date.astype(str) + ' ' + df['arrival_time'].astype(str))
    df['departure_datetime'] = pd.to_datetime(df['start_date'].dt.date.astype(str) + ' ' + df['departure_time'].astype(str))

    df.drop(columns=[
        "arrival_time", "departure_time", "start_date", "end_date", "service_id",
        "stop_sequence", "shape_dist_traveled"
    ], inplace=True)

    # Calculate 'next_stop'
    df = df.sort_values(by=['trip_id', 'arrival_datetime'])
    df["next_stop"] = df.groupby("trip_id")["stop_id"].shift(-1).fillna(-1).astype('int')

    return df

df = clean_and_process_gtfs_data()
df.head(-10)

,route_id,trip_id,route_short_name,route_type,stop_id,stop_name,stop_lat,stop_lon,arrival_datetime,departure_datetime,next_stop
301157,0_104,0_2_1963149,104,3,7053,Metro Bródno,52.293529,21.031043,2025-12-09 06:15:00,2025-12-09 06:15:00,1614
301158,0_104,0_2_1963149,104,3,1614,Metro Bródno,52.293676,21.032621,2025-12-09 06:17:00,2025-12-09 06:17:00,1751
301159,0_104,0_2_1963149,104,3,1751,Suwalska,52.296123,21.032985,2025-12-09 06:18:00,2025-12-09 06:18:00,1750
301160,0_104,0_2_1963149,104,3,1750,Turmoncka,52.298815,21.032966,2025-12-09 06:19:00,2025-12-09 06:19:00,1627
301161,0_104,0_2_1963149,104,3,1627,Kopijników,52.300338,21.031980,2025-12-09 06:20:00,2025-12-09 06:20:00,1628
...,...,...,...,...,...,...,...,...,...,...,...
5846559,6_518,6_2_4827673,518,3,4058,Odkryta,52.324678,20.937698,2025-12-16 22:25:00,2025-12-16 22:25:00,4056
5846560,6_518,6_2_4827673,518,3,4056,Nowodworska,52.320643,20.940360,2025-12-16 22:26:00,2025-12-16 22:26:00,4870
5846561,6_518,6_2_4827673,518,3,4870,Pancera,52.318314,20.946588,2025-12-16 22:27:00,2025-12-16 22:27:00,1677
5846562,6_518,6_2_4827673,518,3,1677,Tarchomin,52.316341,20.952537,2025-12-16 22:28:00,2025-12-16 22:28:00,1676


In [14]:
print(len(df[df['route_type'] == 3]))
print(len(df[df['route_type'] == 2]))
print(len(df[df['route_type'] == 1]))
print(len(df[df['route_type'] == 0]))

4401903
38349
0
1329434
